# LIBERO Absolute Control Test

Test that the environment accepts **absolute EE pose** as actions.
Each action is `[x, y, z, rx, ry, rz, gripper]` in world coordinates —
you send it once via `env.step()` and the env's internal controller drives there.

**Start the ZMQ server first:**
```bash
conda activate libero
python scripts/libero_env_server.py --env libero_spatial
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio
from IPython.display import HTML


from diffusion_policy.env_config import get_env_config
from diffusion_policy.remote_env import RemoteEnv
import diffusion_policy.util.rotation as RotationUtil

## 1. Connect & reset

In [42]:
ENV_KEY = "libero"

cfg = get_env_config(ENV_KEY)
address = cfg.get("zmq_address", "tcp://localhost:5555")
print(f"Connecting to {address}...")

env = RemoteEnv(address=address)
obs = env.reset()

print("Observation keys:")
for k, v in obs.items():
    v = np.asarray(v)
    print(f"  {k}: shape={v.shape}, dtype={v.dtype}")

Connecting to tcp://localhost:5555...
Observation keys:
  robot0_joint_pos: shape=(7,), dtype=float64
  robot0_joint_pos_cos: shape=(7,), dtype=float64
  robot0_joint_pos_sin: shape=(7,), dtype=float64
  robot0_joint_vel: shape=(7,), dtype=float64
  robot0_eef_pos: shape=(3,), dtype=float64
  robot0_eef_quat: shape=(4,), dtype=float64
  robot0_gripper_qpos: shape=(2,), dtype=float64
  robot0_gripper_qvel: shape=(2,), dtype=float64
  agentview_image: shape=(128, 128, 3), dtype=uint8
  robot0_eye_in_hand_image: shape=(128, 128, 3), dtype=uint8
  akita_black_bowl_1_pos: shape=(3,), dtype=float64
  akita_black_bowl_1_quat: shape=(4,), dtype=float64
  akita_black_bowl_1_to_robot0_eef_pos: shape=(3,), dtype=float64
  akita_black_bowl_1_to_robot0_eef_quat: shape=(4,), dtype=float32
  cream_cheese_1_pos: shape=(3,), dtype=float64
  cream_cheese_1_quat: shape=(4,), dtype=float64
  cream_cheese_1_to_robot0_eef_pos: shape=(3,), dtype=float64
  cream_cheese_1_to_robot0_eef_quat: shape=(4,), dtype=

## 2. Read home pose

In [43]:
def get_ee_state(obs):
    pos = np.asarray(obs["robot0_eef_pos"]).flatten()
    quat = np.asarray(obs["robot0_eef_quat"]).flatten()
    gripper = np.asarray(obs["robot0_gripper_qpos"]).flatten()
    return pos, quat, gripper


home_pos, home_quat, home_gripper = get_ee_state(obs)
print(f"Home position:   {home_pos}")
print(f"Home quaternion: {home_quat}")
print(f"Home gripper:    {home_gripper}")


def display_last_frame():
    frame = env.render()
    plt.figure(figsize=(5, 5))
    plt.imshow(frame[::-1])  # flip image vertically
    plt.axis("off")
    plt.show()


Home position:   [-2.08464661e-01 -2.78370651e-14  1.17327948e+00]
Home quaternion: [ 9.99596605e-01  2.46212832e-04 -2.84001205e-02 -6.99529583e-06]
Home gripper:    [ 0.02054187 -0.02054185]


## 3. Helper: send one absolute action

In [44]:
def send(env, obs, pos, ori=(0.0, 0.0, 0.0), gripper=0.0):
    """Send one absolute pose action via env.step(). Returns new obs."""
    action = np.array(list(pos) + list(ori) + [gripper], dtype=np.float32)

    before, _, _ = get_ee_state(obs)
    obs_new, reward, done, info = env.step(action)
    after, quat, grip = get_ee_state(obs_new)

    err = np.linalg.norm(np.array(pos) - after)
    print(f"  sent:   {np.round(pos, 4)}")
    print(f"  before: {np.round(before, 4)}")
    print(f"  after:  {np.round(after, 4)}   err={err:.4f}m")
    print()
    return obs_new

In [ ]:
import torch

imgs = []

obs = env.reset()
home_pos, home_quat, home_gripper = get_ee_state(obs)
home_aa = RotationUtil.quaternion_to_axis_angle(torch.from_numpy(home_quat)).numpy()

print("=== +5cm Z ===")
for i in range(180):
    obs = send(
        env,
        obs,
        pos=home_pos,
        # pos=[0.0, 0.0, 0.0],
        ori=home_aa,
        gripper=home_gripper[0],
    )
    imgs.append(obs["agentview_image"])

video_writer = imageio.get_writer("output.mp4", fps=60)
for image in imgs:
    video_writer.append_data(image[::-1])
video_writer.close()

HTML("""
    <video width="640" height="480" controls>
        <source src="output.mp4" type="video/mp4">
    </video>
    <script>
        var video = document.getElementsByTagName('video')[0];
        video.playbackRate = 2.0; // Increase the playback speed to 2x
        </script>    
""")

=== +5cm Z ===
  sent:   [-0.2085 -0.      1.1733]
  before: [-0.2085 -0.      1.1733]
  after:  [-2.0910e-01  5.0000e-04  1.1728e+00]   err=0.0009m

  sent:   [-0.2085 -0.      1.1733]
  before: [-2.0910e-01  5.0000e-04  1.1728e+00]
  after:  [-2.0990e-01  1.0000e-03  1.1722e+00]   err=0.0021m

  sent:   [-0.2085 -0.      1.1733]
  before: [-2.0990e-01  1.0000e-03  1.1722e+00]
  after:  [-0.2107  0.0014  1.1716]   err=0.0031m

  sent:   [-0.2085 -0.      1.1733]
  before: [-0.2107  0.0014  1.1716]
  after:  [-0.2115  0.0016  1.1711]   err=0.0041m

  sent:   [-0.2085 -0.      1.1733]
  before: [-0.2115  0.0016  1.1711]
  after:  [-0.2122  0.0018  1.1706]   err=0.0050m

  sent:   [-0.2085 -0.      1.1733]
  before: [-0.2122  0.0018  1.1706]
  after:  [-0.213   0.0021  1.1701]   err=0.0060m

  sent:   [-0.2085 -0.      1.1733]
  before: [-0.213   0.0021  1.1701]
  after:  [-0.2139  0.0024  1.1696]   err=0.0071m

  sent:   [-0.2085 -0.      1.1733]
  before: [-0.2139  0.0024  1.1696]
  af